Anomaliedetektion mit Autoencoder

Benötigte Module importieren und Datei laden. Die ersten Zeilen werden ausgegeben.

In [ ]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import mean_absolute_error
from keras.models import Model
import numpy as np

#Pfad für Colab
path = "/content/ekg.csv"
data = pd.read_csv(path, delimiter=',', header=None)
print(data.head())

Daten vorbereiten.

In [ ]:
# entferne erste Spalte
data.drop([0], axis = 1, inplace=True)

# Aufteilung in zwei Tabellen (beide enthalten noch die Zielspalte Class)
train_data, test_data = train_test_split(data, test_size=0.3, random_state=42)

# trainiert wird nur mit normalen Daten
train_data = train_data[train_data[141]==1]
# entferne Zielspalte, wird nicht benötigt
train_data.drop([141], axis = 1, inplace=True)

# von den Testdaten wird die Zielspalte entfernt und in einer Variablen gespeichert
test_col = test_data[141]
test_data.drop([141], axis = 1, inplace=True)

print(train_data.shape)
print(test_data.shape)

Autoencoder aufbauen

In [ ]:
encoder = tf.keras.Sequential(name='encoder')
encoder.add(tf.keras.Input(shape=(140,)))
encoder.add(layer=tf.keras.layers.Dense(units=64, activation=tf.nn.sigmoid))
encoder.add(layer=tf.keras.layers.Dense(units=32, activation=tf.nn.sigmoid))
encoder.add(layer=tf.keras.layers.Dense(units=8, activation=tf.nn.sigmoid))

decoder = tf.keras.Sequential(name='decoder')
decoder.add(tf.keras.Input(shape=(8,)))
decoder.add(layer=tf.keras.layers.Dense(units=32, activation=tf.nn.sigmoid))
decoder.add(layer=tf.keras.layers.Dense(units=64, activation=tf.nn.sigmoid))
decoder.add(layer=tf.keras.layers.Dense(units=140, activation=tf.nn.sigmoid))

autoencoder = tf.keras.Sequential([encoder, decoder], name='autoencoder')

autoencoder.compile(optimizer='adam', loss='mae', metrics=['mae'])

encoder.summary()
decoder.summary()
autoencoder.summary()

Training

In [ ]:
cb_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

autoencoder.fit(train_data, train_data, validation_data=(test_data, test_data), epochs=100, batch_size=16, callbacks=[cb_early])

Threshold berechnen, nur mit den "normalen" Trainingsdaten

In [ ]:
train_pred = autoencoder.predict(train_data)
threshold = mean_absolute_error(train_pred, train_data)

print("Threshold:", threshold)

Spalte mit vorhersagen generieren, mit komplette Testdaten

In [ ]:
new_threshold = 0.55

test_pred = autoencoder.predict(test_data)
maes = tf.keras.losses.mae(test_data, test_pred)
pred_col_bool = tf.math.less(maes, new_threshold)
pred_col = tf.cast(pred_col_bool, dtype=tf.int32)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(test_col, pred_col)

disp = ConfusionMatrixDisplay(cm)

disp.plot()